# 01 Molecules as Data

This section starts from SMILES. SMILES means **Simplified Molecular Input Line Entry System**. It is a way to write a molecular graph as a one-line string. SMILES is convenient for storage, search, tables, descriptors, fingerprints, and machine-learning input.

Application scenarios include chemical database search, structure deduplication, virtual screening, property prediction, and reaction SMILES representation. The classic origin is Weininger's SMILES notation; in practice, parsing behavior follows toolkits such as RDKit and Daylight.

## Intuition

SMILES can be understood as "walking through a molecular graph and writing down atoms, bonds, branches, and ring closures along the path". The same molecular graph can be traversed from different atoms, in different directions, and with different branch orders. Therefore, a single molecule often has multiple valid SMILES strings.

Example: ethanol can be written as `CCO` or `OCC`. Both represent the same molecular graph.

For deduplication and comparison, input SMILES are often converted to canonical SMILES. The software first parses the molecular graph, then uses fixed ranking and traversal rules to choose a standard string. If the molecular graph, isotopes, charges, aromaticity, stereochemistry, and related information are interpreted by the same toolkit under the same rules, the output should be deterministic.

In [ ]:
from pathlib import Path

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find the materials root. Start Jupyter from the materials directory "
        "or from one of its subdirectories."
    )

DATA = ROOT / "data"
RAW = DATA / "raw"
EXAMPLES = DATA / "examples"
RANDOM_STATE = 42

print("materials root:", ROOT)

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

# This cell reads the example molecule table. Later each molecule is parsed by RDKit into a molecule object.
examples = pd.read_csv(EXAMPLES / "example_molecules.csv")
examples

## Mathematical and Chemical Definitions

A molecular graph can be written as:

```text
G = (V, E)
V: atoms
E: bonds
```

SMILES is one linearization of this graph, not the only one. canonical SMILES is an algorithmically selected standard linearization.

Important caveat: the "uniqueness" of canonical SMILES is practical, not absolute outside a chosen software toolkit and chemical convention. If stereochemistry, protonation state, salt form, tautomeric state, or atom mapping is not specified, two strings may still hide chemical distinctions beyond the scope of this tutorial.

In [ ]:
def parse_status(smiles):
    # MolFromSmiles parses a string into an RDKit molecular graph; failed parsing returns None.
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return pd.Series({"valid": False, "canonical SMILES": None, "atoms": None, "bonds": None})
    return pd.Series(
        {
            "valid": True,
            "canonical SMILES": Chem.MolToSmiles(mol),
            "atoms": mol.GetNumAtoms(),
            "bonds": mol.GetNumBonds(),
        }
    )

parsed = pd.concat([examples, examples["smiles"].apply(parse_status)], axis=1)
parsed

## Code

RDKit can parse SMILES into molecule objects and draw 2D structures. Invalid SMILES returns `None`.

In [ ]:
valid = parsed[parsed["valid"]].copy()
mols = [Chem.MolFromSmiles(smi) for smi in valid["smiles"]]
legends = [f"{name}\n{can}" for name, can in zip(valid["name"], valid["canonical SMILES"])]

# SVG is a vector format and remains clear when zoomed; larger subImgSize makes legend SMILES easier to read.
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(360, 260), legends=legends, useSVG=True)

In [ ]:
student_smiles = "CC(=O)Oc1ccccc1C(=O)O"  # try replacing this with your molecule
mol = Chem.MolFromSmiles(student_smiles)

if mol is None:
    print("RDKit cannot parse this SMILES.")
else:
    print("canonical SMILES:", Chem.MolToSmiles(mol))
    print("atoms:", mol.GetNumAtoms(), "bonds:", mol.GetNumBonds())

    mol_with_labels = Chem.Mol(mol)
    for atom in mol_with_labels.GetAtoms():
        atom.SetProp("atomNote", f"{atom.GetSymbol()}{atom.GetIdx()}")

    display(
        Draw.MolsToGridImage(
            [mol_with_labels],
            molsPerRow=1,
            subImgSize=(520, 380),
            legends=["atom labels show element symbol + atom index"],
            useSVG=True,
        )
    )

## Observation Questions

1. Do `CCO` and `OCC` have the same canonical SMILES?
2. Why can RDKit not parse `invalid_ring`?
3. What problem does canonical SMILES solve, and what does it not solve?

### Hints

1. Parse `CCO` and `OCC` separately, then compare `Chem.MolToSmiles(...)`.
2. Common invalid-SMILES causes include unclosed ring numbers, mismatched parentheses, unreasonable valence, and illegal characters.
3. canonical SMILES mainly solves deduplication and comparison when the same molecule has multiple string forms. It does not automatically solve tautomerism, protonation state, salt form, conformation, solvent, or experimental context.

## Summary

SMILES lets molecules enter tables and code; canonical SMILES makes deduplication easier. But SMILES is not all chemical information. Experimental conditions, conformations, solvents, and measurement errors still need separate records.